# Parcial para la casa · Raíces C — ¿Cuánto te cobra de verdad el banco?

**Métodos Numéricos · UNAL Manizales · 2026-2**

Un crédito de monto P que se paga en n cuotas mensuales iguales A tiene una tasa mensual i que cumple

$$P = A\sum_{k=1}^{n}(1+i)^{-k} = A\,\frac{1-(1+i)^{-n}}{i}.$$

Si te dicen P, n y A (lo que te muestra el simulador del banco), **no hay fórmula para despejar i**: es un problema de raíces. En este cuaderno lo resuelves con **regla falsa**, **secante** y **Newton**, usando tasas reales que reportan los bancos colombianos a la Superintendencia Financiera, y comparas con la **tasa de usura** vigente.

**Fuentes de los datos (abiertas, sin clave):**
1. *Tasas de interés activas por tipo de crédito – Histórico*, Superintendencia Financiera de Colombia (SFC), datos.gov.co, dataset `w9zh-vetq`: `https://www.datos.gov.co/resource/w9zh-vetq.json`. Usamos el corte semanal **31 de julio de 2026** (el más reciente del histórico) y la columna `tasa_efectiva_promedio` (tasa efectiva anual, %), **ponderada por monto desembolsado** por entidad.
2. *Tasa de interés bancario corriente – TIBC*, SFC, datos.gov.co, dataset `pare-7x5i`. La **usura** es 1,5 veces el interés bancario corriente (Código Penal, art. 305); la SFC la publica junto con la resolución (Resolución 1472 de 2026: IBC 19,06 % E.A. y usura 28,59 % E.A. para consumo y ordinario, del 1 al 31 de octubre de 2026).

**Fecha de consulta de la copia congelada:** 15 de octubre de 2026.
**Licencia:** Creative Commons Atribución–CompartirIgual 4.0 Internacional (así figura en datos.gov.co para ambos datasets). Cita: "Superintendencia Financiera de Colombia, datos.gov.co, consultado el 15/10/2026".

Reglas del cuaderno: solo cambias lo marcado `# <-- CAMBIA ESTO`. Antes de cada experimento hay una caja **Predice en tu guía antes de correr**: escribe tu predicción en papel *antes* de ejecutar.

## Celda 1 · Carga de datos reales (API de datos.gov.co + respaldo congelado)

La celda le pide a la API Socrata de datos.gov.co, para el corte del 31/07/2026, el monto desembolsado y la tasa ponderada **por entidad** en tres productos:
- **Libre inversión** (crédito de consumo),
- **Vivienda**: adquisición de vivienda no VIS en pesos,
- **Tarjeta de crédito**: compras diferidas entre 25 y 36 meses (incluye tarjetas de personas y empresariales; filtro por la columna `plazo_de_cr_dito`).

Y le pide al dataset del interés bancario corriente los valores de julio y octubre de 2026. Si no hay internet o la API falla, usa la copia congelada (los mismos números, copiados textualmente el 15/10/2026).

In [ ]:
import urllib.parse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

URL_TASAS = "https://www.datos.gov.co/resource/w9zh-vetq.json"
URL_IBC   = "https://www.datos.gov.co/resource/pare-7x5i.json"
CORTE = "2026-07-31T00:00:00.000"

FILTROS = {
    "libre":    "producto_de_cr_dito = 'Libre inversión'",
    "vivienda": "producto_de_cr_dito = 'Adquisición de vivienda no vis (colocación en pesos)'",
    "tarjeta":  "plazo_de_cr_dito like '%25 y 36%'",
}

def consulta_socrata(url, params):
    return pd.read_json(url + "?" + urllib.parse.urlencode(params, quote_via=urllib.parse.quote))

def tasas_por_entidad(filtro):
    params = {"$select": "nombre_entidad, sum(montos_desembolsados) as monto, "
                         "sum(tasa_efectiva_promedio*montos_desembolsados) as tasa_por_monto",
              "$where": f"fecha_corte = '{CORTE}' AND {filtro}",
              "$group": "nombre_entidad", "$limit": 1000}
    df = consulta_socrata(URL_TASAS, params)
    return [(r.nombre_entidad, float(r.monto), float(r.tasa_por_monto)) for r in df.itertuples()]

# ---- Copia congelada (WebFetch textual del 15/10/2026): (entidad, monto, suma de tasa*monto) ----
libre = [
    ('Lulo Bank', 18777702650.19, 406598092592.9104),
    ('BBVA Colombia', 88891491381.93, 1955648479322.266),
    ('Banco Falabella S.A.', 70817692049.0, 1663120295883.08),
    ('Banco de Occidente', 69454143490.35, 1448912359202.3806),
    ('Banco Davibank', 65463979173.0, 1532246367401.39),
    ('Banco Mundo Mujer S.A.', 5799412000.0, 164810185120.0),
    ('Rappipay', 10122154800.0, 253219834002.0),
    ('La Hipotecaria', 75013206.0, 1741271391.8),
    ('Tuya', 1888850000.0, 48743500000.0),
    ('Crezcamos', 407856000.0, 10262046600.0),
    ('JFK Cooperativa Financiera', 20543029051.0, 418964158905.87),
    ('Credifamilia', 562196063.0, 11144037617.92),
    ('Confiar', 8215783458.0, 140941126704.25),
    ('Cotrafa', 7841929975.0, 169285741336.26),
    ('Coltefinanciera', 394859497.0, 10708518537.01),
    ('Banco Davivienda', 187224862021.0, 4377572368563.76),
    ('Bancolombia', 195365193074.4, 4132680350711.3296),
    ('Banco Caja Social S.A.', 26112102000.0, 613982859000.0),
    ('AV Villas', 51500892881.0, 1019755657612.02),
    ('Banco Serfinanza S.A.', 4463476381.89, 97317543247.4403),
    ('Financiera Juriscoop C.F.', 345482880.0, 9520289246.4),
    ('Bancoomeva', 15755311145.0, 362434431436.52),
    ('Cooperativa Financiera de Antioquia', 2028795180.0, 54321141730.79),
    ('Finandina', 1497824211.0, 42038072625.86),
    ('Itaú', 16654209451.4, 306058126128.56),
    ('Coopcentral', 166000000.0, 2750000000.0),
    ('Banagrario', 386272452.0, 9659274614.64),
    ('Nu Colombia C.F.', 8143333214.04, 167093800475.8326),
    ('Banco GNB Sudameris', 3257000000.0, 59229800000.0),
    ('Banco de Bogotá', 218261232092.0, 5122842387505.67),
    ('Banco Unión', 291119599.0, 8371948107.4),
    ('Banco Popular', 18300000.0, 460794000.0),
]
vivienda = [
    ('AV Villas', 21031999318.0, 286835834049.86),
    ('Banagrario', 3133552794.0, 44594199549.4),
    ('Banco Caja Social S.A.', 24680294654.0, 364638993146.32),
    ('Banco Davibank', 17827718460.0, 270304149004.72),
    ('Banco Davivienda', 70835907551.0, 1107734661430.39),
    ('Banco de Bogotá', 53514966576.0, 847690782309.08),
    ('Banco de Occidente', 30339961472.0, 454923718470.84),
    ('Bancolombia', 72803381841.0, 1147373302897.82),
    ('Bancoomeva', 2660386000.0, 35308174599.2),
    ('Banco Unión', 1258734210.0, 22657215780.0),
    ('BBVA Colombia', 21066443357.68, 331321939832.4889),
    ('Confiar', 1572544811.0, 23428172165.0),
    ('Cotrafa', 210000000.0, 3521700000.0),
    ('Credifamilia', 56000000.0, 943040000.0),
    ('Fondo Nacional del Ahorro', 13773156975.14, 165410817743.5197),
    ('Itaú', 17191927271.35, 235007737079.126),
]
tarjeta = [
    ('AV Villas', 7584887572.43, 205612723862.9212),
    ('Banagrario', 118605316.56, 2892512021.57),
    ('Bancien', 20009841.0, 510078674.16),
    ('Banco Caja Social S.A.', 2165716382.0, 59526149545.08),
    ('Banco Davibank', 31866714370.0, 900067979085.56),
    ('Banco Davivienda', 31734568790.15, 895401081864.5848),
    ('Banco de Bogotá', 49233408127.0, 1343520018664.98),
    ('Banco de Occidente', 2288482781.0, 49849008254.35),
    ('Banco Falabella S.A.', 8691365814.85, 247567090670.931),
    ('Banco GNB Sudameris', 1120483746.0, 29713860953.39),
    ('Bancolombia', 86053483218.12, 2471359758340.024),
    ('Bancoomeva', 478966272.2, 12164917224.1809),
    ('Banco Popular', 1823768423.63, 52379797855.0089),
    ('Banco Serfinanza S.A.', 488290207.0, 12829123189.99),
    ('Banco Unión', 770045.0, 22154194.65),
    ('Banco W S.A.', 2077519.0, 58773012.51),
    ('BBVA Colombia', 7795698788.69, 203133462009.6348),
    ('Coltefinanciera', 84003372.0, 2405078650.95),
    ('Confiar', 9340686.0, 247808399.58),
    ('Coopcentral', 5055798.0, 92875009.26),
    ('Financiera Juriscoop C.F.', 11249326.0, 318700955.12),
    ('Finandina', 1698784828.75, 47848772857.6883),
    ('Itaú', 1314296762.89, 37530643054.4354),
    ('JFK Cooperativa Financiera', 38515376.0, 1021427771.52),
    ('Lulo Bank', 271026927.55, 6489019760.919),
    ('Nu Colombia C.F.', 11828226364.55, 333516372022.9112),
    ('Tuya', 5419340549.0, 151562575680.0),
]

RESPALDO = {"libre": libre, "vivienda": vivienda, "tarjeta": tarjeta}
IBC_RESPALDO = {"2026-07": 19.19, "2026-10": 19.06}   # % E.A., consumo y ordinario (Res. 0965 y 1472 de 2026)

tablas, origen = {}, {}
for prod, filtro in FILTROS.items():
    try:
        filas = tasas_por_entidad(filtro); assert len(filas) > 5
        origen[prod] = "API en vivo"
    except Exception as err:
        filas = RESPALDO[prod]; origen[prod] = "respaldo congelado"
    df = pd.DataFrame(filas, columns=["entidad", "monto", "tasa_por_monto"])
    df["tasa_EA_%"] = df["tasa_por_monto"] / df["monto"]          # tasa ponderada por monto
    tablas[prod] = df[["entidad", "monto", "tasa_EA_%"]].sort_values("monto", ascending=False).set_index("entidad")

ibc = {}
for mes in ["2026-07", "2026-10"]:
    try:
        d = consulta_socrata(URL_IBC, {"vigencia_desde": mes + "-01T00:00:00.000", "modalidad": "CONSUMO Y ORDINARIO"})
        ibc[mes] = float(str(d["interes_bancario_corriente"].iloc[0]).replace("%", "")); fuente_ibc = "API en vivo"
    except Exception as err:
        ibc[mes] = IBC_RESPALDO[mes]; fuente_ibc = "respaldo congelado"
usura = {mes: round(1.5*v, 2) for mes, v in ibc.items()}       # usura = 1,5 x IBC (Código Penal, art. 305)

for prod in tablas:
    t = tablas[prod]
    pond = (t["tasa_EA_%"]*t["monto"]).sum()/t["monto"].sum()
    print(f"{prod:9s}: {len(t):2d} entidades, tasa E.A. entre {t['tasa_EA_%'].min():.2f}% y {t['tasa_EA_%'].max():.2f}%, "
          f"promedio ponderado {pond:.2f}%   [{origen[prod]}]")
print(f"Interés bancario corriente (consumo y ordinario): julio {ibc['2026-07']}%, octubre {ibc['2026-10']}%   [{fuente_ibc}]")
print(f"Tasa de usura = 1,5 x IBC: julio {usura['2026-07']}%, octubre {usura['2026-10']}%")
tablas["libre"].head(8)

## Celda 2 · Las herramientas

- Tasa efectiva anual → mensual: i = (1 + EA)^(1/12) − 1. Mensual → anual: EA = (1 + i)¹² − 1.
- Cuota: A = P·i / (1 − (1 + i)⁻ⁿ).
- La función a la que le buscamos raíz (escrita **como suma**, sin dividir por i):

$$g(i) = P - A\sum_{k=1}^{n}(1+i)^{-k}, \qquad g'(i) = A\sum_{k=1}^{n}k\,(1+i)^{-k-1}.$$

- Cada método devuelve la raíz, las iteraciones, las **evaluaciones** (cuántas veces calculó g o g′) y la historia.

**Para completar (opcional, como en los labs):** escribe el paso de la regla falsa en `paso_regla_falsa`: la raíz de la recta que pasa por (a, g(a)) y (b, g(b)). Si lo dejas en `None`, la Celda 2b usa la versión de respaldo.

In [ ]:
def mensual(EA):            # EA en fracción (0.2115 = 21,15 %)
    return (1 + EA)**(1/12) - 1

def anual(i):
    return (1 + i)**12 - 1

def cuota(P, n, i):
    return P*i/(1 - (1 + i)**(-n))

def hacer_g(P, n, A):
    """Devuelve g(i) = P - A*sum (1+i)^(-k) y su derivada (forma de SUMA, bien definida en i = 0)."""
    k = np.arange(1, n + 1)
    g  = lambda i: P - A*np.sum((1.0 + i)**(-k))
    gp = lambda i: A*np.sum(k*(1.0 + i)**(-k - 1.0))
    return g, gp

def paso_regla_falsa(a, b, ga, gb):
    # COMPLETA: devuelve la raíz de la recta que pasa por (a, ga) y (b, gb)
    return None  # <-- CAMBIA ESTO

def regla_falsa(g, a, b, tol=1e-10, maxit=500):
    """Regla falsa en [a, b] (necesita g(a)*g(b) < 0). Para cuando |c_k - c_{k-1}| < tol.
    Devuelve (raíz, iteraciones, evaluaciones, historia de (a, b, c))."""
    ga, gb = g(a), g(b); ev = 2; hist = []; c_ant = None
    for k in range(1, maxit + 1):
        c = paso_regla_falsa(a, b, ga, gb)
        hist.append((a, b, c))
        if c_ant is not None and abs(c - c_ant) < tol:
            return c, k, ev, hist
        gc = g(c); ev += 1
        if gc == 0:
            return c, k, ev, hist
        if ga*gc < 0:
            b, gb = c, gc       # la raíz está en [a, c]
        else:
            a, ga = c, gc       # la raíz está en [c, b]
        c_ant = c
    return c, None, ev, hist

def secante(g, x0, x1, tol=1e-10, maxit=100):
    """Secante desde x0, x1. Para cuando |x_{k+1} - x_k| < tol."""
    g0, g1 = g(x0), g(x1); ev = 2; hist = [x0, x1]
    for k in range(1, maxit + 1):
        if g1 == g0:
            return x1, None, ev, hist
        x2 = x1 - g1*(x1 - x0)/(g1 - g0); hist.append(x2)
        if not np.isfinite(x2) or x2 <= -1:
            return x2, None, ev, hist
        if abs(x2 - x1) < tol:
            return x2, k, ev, hist
        x0, g0 = x1, g1; x1 = x2; g1 = g(x1); ev += 1
    return x1, None, ev, hist

def newton(g, gp, x0, tol=1e-10, maxit=100):
    """Newton desde x0. Para cuando |x_{k+1} - x_k| < tol. Cada iteración cuesta 2 evaluaciones (g y g')."""
    x = x0; hist = [x0]; ev = 0
    for k in range(1, maxit + 1):
        gx, gpx = g(x), gp(x); ev += 2
        if not (np.isfinite(gx) and np.isfinite(gpx)) or gpx == 0:
            return x, None, ev, hist
        xn = x - gx/gpx; hist.append(xn)
        if not np.isfinite(xn) or xn <= -1:
            return xn, None, ev, hist
        if abs(xn - x) < tol:
            return xn, k, ev, hist
        x = xn
    return x, None, ev, hist

### Celda 2b · Respaldo del paso de la regla falsa (no la modifiques)

In [ ]:
if paso_regla_falsa(0.0, 1.0, -1.0, 3.0) is None:
    def paso_regla_falsa(a, b, ga, gb):
        return b - gb*(b - a)/(gb - ga)
    print("Usando el paso de la regla falsa de respaldo.")
else:
    print("Tu paso desde (0,-1) y (1,3) da", paso_regla_falsa(0.0, 1.0, -1.0, 3.0), "(debe dar 0.25)")

## Celda 3 · Ítem II.1 — Del banco a la cuota, y de la cuota de vuelta a la tasa

Tomamos la tasa real de una entidad, construimos la cuota A de un crédito de P pesos a n meses y luego **olvidamos la tasa**: solo con (P, n, A) la recuperamos con los tres métodos. Puntos iniciales: regla falsa en [0, A/P]; secante con x₀ = 0, x₁ = A/P; Newton con i₀ = 0.

**Predice en tu guía antes de correr:**
(a) Con la tasa de Bancolombia en libre inversión (≈ 21,15 % E.A.), ¿de cuánto será la cuota de un crédito de 10 millones a 36 meses? (Una cuenta rápida: 10 millones/36 ≈ 277.800 sin intereses.)
(b) Ordena los tres métodos de menos a más iteraciones. ¿Cuál crees que gasta menos **evaluaciones**?

In [ ]:
producto = "libre"        # <-- CAMBIA ESTO: "libre", "vivienda" o "tarjeta"
entidad  = "Bancolombia"  # <-- CAMBIA ESTO (mira los nombres en la Celda 1)
P   = 10_000_000          # <-- CAMBIA ESTO (pesos)
n   = 36                  # <-- CAMBIA ESTO (meses)
tol = 1e-10               # <-- CAMBIA ESTO

EA = tablas[producto].loc[entidad, "tasa_EA_%"]/100
i_real = mensual(EA)
A = cuota(P, n, i_real)
print(f"{entidad}, {producto}: tasa publicada {100*EA:.4f}% E.A.  ->  i mensual = {i_real:.10f}")
print(f"Crédito de {P:,.0f} a {n} meses: cuota A = {A:,.2f} pesos (total pagado {n*A:,.0f})")

g, gp = hacer_g(P, n, A)
dAdi = (cuota(P, n, i_real + 1e-7) - cuota(P, n, i_real - 1e-7))/2e-7
print(f"Sensibilidad: dA/di ≈ {dAdi:,.0f} pesos  ->  un centavo de cuota equivale a un cambio en i de {0.01/dAdi:.2e}")
print(f"\ng(0) = {g(0):,.2f}    g(A/P) = {g(A/P):,.2f}    (A/P = {A/P:.6f})")
resultados = {
    "regla falsa [0, A/P]": regla_falsa(g, 0.0, A/P, tol),
    "secante (0, A/P)":     secante(g, 0.0, A/P, tol),
    "Newton (i0 = 0)":      newton(g, gp, 0.0, tol),
}
filas = []
for nombre, (x, it, ev, h) in resultados.items():
    filas.append(dict(metodo=nombre, iteraciones=it, evaluaciones=ev, i=x,
                      error_en_i=abs(x - i_real), EA_recuperada_pct=100*anual(x)))
pd.DataFrame(filas).set_index("metodo")

## Celda 4 · Ítem II.2 — El extremo fijo de la regla falsa y el orden de convergencia

**Predice en tu guía antes de correr:**
(a) En la tabla de la regla falsa, ¿cuál de los dos extremos, a o b, crees que se mueve? ¿Los dos?
(b) ¿En qué columna esperas una constante: `razon_lineal` = eₖ₊₁/eₖ o `razon_cuadratica` = eₖ₊₁/eₖ²? Para cada método.
(c) Si cambias a vivienda a 360 meses (30 años), ¿la regla falsa necesitará más o menos iteraciones que a 36 meses?

In [ ]:
producto = "libre"        # <-- CAMBIA ESTO
entidad  = "Bancolombia"  # <-- CAMBIA ESTO
P, n = 10_000_000, 36     # <-- CAMBIA ESTO (prueba producto="vivienda", n=360)

i_real = mensual(tablas[producto].loc[entidad, "tasa_EA_%"]/100)
A = cuota(P, n, i_real); g, gp = hacer_g(P, n, A)

x, it, ev, hist = regla_falsa(g, 0.0, A/P)
rf = pd.DataFrame(hist, columns=["a", "b", "c"])
rf["error"] = (rf["c"] - i_real).abs()
rf["razon_lineal"] = rf["error"].shift(-1)/rf["error"]
print(f"Regla falsa: {it} iteraciones. ¿El extremo a se movió alguna vez? {'sí' if (rf['a'] != 0).any() else 'NO: a = 0 siempre'}")
print(rf.head(10).to_string(float_format=lambda v: f"{v:.3e}"))
k = np.arange(1, n + 1)
gpp = -A*np.sum(k*(k + 1)*(1.0 + i_real)**(-k - 2.0))
pend_tangente = gp(i_real); pend_cuerda = (0 - g(0.0))/i_real
print(f"\nPendiente de la tangente en la raíz g'(i*) = {pend_tangente:,.0f};  pendiente de la cuerda de (0, g(0)) a (i*, 0) = {pend_cuerda:,.0f}")
print(f"Constante teórica de la regla falsa  C = 1 - tangente/cuerda = {1 - pend_tangente/pend_cuerda:.4f}")
print(f"Constante teórica de Newton  |g''(i*)|/(2 g'(i*)) = {abs(gpp)/(2*pend_tangente):.4f}")

def tabla_errores(h, i_real):
    e = np.abs(np.array(h[1:], dtype=float) - i_real)     # errores desde el primer iterado nuevo
    t = pd.DataFrame({"error": e})
    t["razon_lineal"] = t["error"].shift(-1)/t["error"]
    t["razon_cuadratica"] = t["error"].shift(-1)/t["error"]**2
    return t[t["error"] > 1e-15]

print("\nSecante (0, A/P):"); print(tabla_errores(secante(g, 0.0, A/P)[3][1:], i_real).to_string(float_format=lambda v: f"{v:.3e}"))
print("\nNewton (i0 = 0):");  print(tabla_errores(newton(g, gp, 0.0)[3], i_real).to_string(float_format=lambda v: f"{v:.3e}"))

# Figura: g(i) y las rectas de la regla falsa, todas ancladas en (0, g(0))
ii = np.linspace(0, A/P, 300)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ii, [g(t)/1e6 for t in ii], color="#2a78d6", lw=2, label="g(i)  (millones de pesos)")
for a_, b_, c_ in hist[:4]:
    ax.plot([a_, b_], [g(a_)/1e6, g(b_)/1e6], color="#eb6834", lw=1)
    ax.plot([c_], [0], "o", color="#eb6834", ms=6)
ax.axhline(0, color="#888", lw=0.8); ax.axvline(i_real, color="#888", lw=0.8, ls=":")
ax.set_xlabel("i (tasa mensual)"); ax.set_ylabel("millones de pesos")
ax.set_title(f"Regla falsa: las cuerdas salen todas de (0, g(0))  ·  {entidad}, n = {n}")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

## Celda 5 · Ítem II.3 — El crédito "sin intereses": i → 0 y la indeterminación 0/0

La fórmula cerrada a(i) = (1 − (1 + i)⁻ⁿ)/i y la suma a(i) = Σ (1 + i)⁻ᵏ son **la misma función** en exactas para i ≠ 0. En i = 0 la cerrada es 0/0; la suma da n.

**Predice en tu guía antes de correr:**
(a) Para i = 10⁻⁶, 10⁻¹⁰ y 10⁻¹⁵, ¿cuántas cifras correctas crees que tendrá la fórmula cerrada? ¿Y en i = 10⁻¹⁶?
(b) Un almacén ofrece 1.200.000 pesos en 12 cuotas de 100.000. ¿Cuál es la tasa? ¿Qué le pasará a Newton si usa la fórmula cerrada?

In [ ]:
n = 36   # <-- CAMBIA ESTO
filas = []
for i in [1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12, 1e-14, 1e-15, 1e-16, 0.0]:
    i = np.float64(i)
    cerrada = (1 - (1 + i)**(-n))/i if i != 0 else np.nan
    suma = np.sum((1 + i)**(-np.arange(1, n + 1.0)))
    filas.append(dict(i=i, a_cerrada=cerrada, a_suma=suma, error_relativo=abs(cerrada - suma)/suma))
print(pd.DataFrame(filas).to_string(float_format=lambda v: f"{v:.15g}"))

# Crédito "sin intereses"
P0, n0, A0 = 1_200_000, 12, 100_000   # <-- CAMBIA ESTO
g_suma, gp_suma = hacer_g(P0, n0, A0)
g_cerr  = lambda i: P0 - A0*(1 - (1 + i)**(-n0))/i
gp_cerr = lambda i: A0*((1 - (1 + i)**(-n0))/i**2 - n0*(1 + i)**(-n0 - 1)/i)
print(f"\ng con la suma en i = 0: {g_suma(0.0)}     g con la fórmula cerrada en i = 0: {g_cerr(np.float64(0.0))}")
tol0 = 1e-12                          # <-- CAMBIA ESTO (prueba 1e-10 y 1e-12)
for i0 in [0.0, A0/P0]:
    for nombre, (gg, ggp) in {"forma de SUMA": (g_suma, gp_suma), "fórmula CERRADA": (g_cerr, gp_cerr)}.items():
        x, it, ev, h = newton(gg, ggp, np.float64(i0), tol=tol0)
        estado = f"convergió en {it} iteraciones" if it is not None else f"NO convergió ({len(h)-1} iterados)"
        print(f"Newton desde i0 = {i0:.4f} con la {nombre}: {estado}; último iterado = {x:.3e}")
        print("   primeros iterados:", ", ".join(f"{v:.2e}" for v in h[:10]))

## Celda 6 · Ítem II.4 — Una oferta: ¿es legal?

Te ofrecen (escenario hipotético) un crédito de 10.000.000 a 36 cuotas de 400.000. La tasa de usura de octubre de 2026 para consumo y ordinario la calculó la Celda 1.

**Predice en tu guía antes de correr:** ¿la tasa efectiva anual queda por encima o por debajo de la usura? ¿Por cuánto? ¿Qué tolerancia en i necesitas para decidir con seguridad?

In [ ]:
P, n, A = 10_000_000, 36, 400_000   # <-- CAMBIA ESTO
mes_usura = "2026-10"                # <-- CAMBIA ESTO ("2026-07" o "2026-10")

g, gp = hacer_g(P, n, A)
i_oferta, it, ev, _ = newton(g, gp, 0.0)
EA_oferta = anual(i_oferta)
us = usura[mes_usura]/100
i_usura = mensual(us)
print(f"Tasa de la oferta: i = {i_oferta:.10f} mensual = {100*EA_oferta:.4f}% E.A.  (Newton, {it} iteraciones)")
print(f"Usura {mes_usura}: {100*us:.2f}% E.A. = {i_usura:.10f} mensual")
print("Veredicto:", "SUPERA la usura" if EA_oferta > us else "no supera la usura",
      f"(diferencia {100*(EA_oferta - us):+.4f} puntos de E.A.)")
print(f"Cuota máxima legal para este P y n: {cuota(P, n, i_usura):,.2f} pesos")
print(f"\nDatos reales (corte 31/07/2026) contra la usura de julio ({usura['2026-07']}%):")
for prod, t in tablas.items():
    print(f"   {prod:9s}: tasa ponderada más alta {t['tasa_EA_%'].max():.2f}% ({t['tasa_EA_%'].idxmax()})")
print(f"\nSensibilidades en la raíz:  dEA/di = 12(1+i)^11 = {12*(1 + i_oferta)**11:.3f}")
h = 1e-7
print(f"                           dA/di ≈ {(cuota(P, n, i_oferta + h) - cuota(P, n, i_oferta - h))/(2*h):,.0f} pesos por unidad de i")
print(f"                           g'(i*) = {gp(i_oferta):,.0f} pesos por unidad de i")

## Celda 7 · Parte III — Auditar al agente

Esta celda recalcula la tabla del agente: para cada entidad de un producto y cada plazo, construye la cuota de un crédito de 10 millones con la tasa real, y mide cuántas evaluaciones gasta cada acción para recuperar i con tolerancia 10⁻¹⁰. También cuenta cosas que te sirven para decidir V o F.

**Predice en tu guía antes de correr:** para cada afirmación de la Parte III que vayas a auditar, escribe primero V o F y por qué; luego córrela con los parámetros que la ponen a prueba.

In [ ]:
producto = "vivienda"             # <-- CAMBIA ESTO: "vivienda", "libre" o "tarjeta"
plazos   = [120, 180, 240, 360]   # <-- CAMBIA ESTO (libre: [12, 24, 36, 60, 84]; tarjeta: [6, 12, 24, 36])
P        = 10_000_000             # <-- CAMBIA ESTO
tol      = 1e-10                  # <-- CAMBIA ESTO

acciones = {
    "regla falsa [0, A/P]": lambda g, gp, b: regla_falsa(g, 0.0, b, tol),
    "secante (0, A/P)":     lambda g, gp, b: secante(g, 0.0, b, tol),
    "Newton i0 = 0":        lambda g, gp, b: newton(g, gp, 0.0, tol),
    "Newton i0 = A/P":      lambda g, gp, b: newton(g, gp, b, tol),
}
reg = []
for entidad, fila in tablas[producto].iterrows():
    i_real = mensual(fila["tasa_EA_%"]/100)
    for n in plazos:
        A = cuota(P, n, i_real); g, gp = hacer_g(P, n, A)
        for nombre, accion in acciones.items():
            x, it, ev, h = accion(g, gp, A/P)
            reg.append(dict(entidad=entidad, n=n, accion=nombre, evaluaciones=ev, iteraciones=it,
                            error=abs(x - i_real), convergio=it is not None and abs(x - i_real) < 1e-9,
                            se_paso_a_la_derecha=any(v > i_real*(1 + 1e-12) for v in h[1:]) if "Newton" in nombre else None,
                            primer_iterado=h[1] if "Newton" in nombre else None))
reg = pd.DataFrame(reg)
print(f"{producto}: {tablas[producto].shape[0]} entidades x {len(plazos)} plazos = {tablas[producto].shape[0]*len(plazos)} créditos")
resumen = reg.groupby("accion").agg(evaluaciones_promedio=("evaluaciones", "mean"), max_eval=("evaluaciones", "max"),
                                    fallos=("convergio", lambda s: (~s).sum()), error_maximo=("error", "max"))
print(resumen.to_string(float_format=lambda v: f"{v:.4g}"))
print("\nEvaluaciones promedio por plazo:")
print(reg.pivot_table(index="n", columns="accion", values="evaluaciones", aggfunc="mean").round(2).to_string())
nw0 = reg[reg.accion == "Newton i0 = 0"]; nwb = reg[reg.accion == "Newton i0 = A/P"]
print(f"\nNewton i0 = 0: créditos donde algún iterado quedó a la DERECHA de la raíz: {int(nw0.se_paso_a_la_derecha.sum())}")
print(f"Newton i0 = A/P: primer iterado más pequeño: {nwb.primer_iterado.min():.4f}  "
      f"(créditos con primer iterado negativo: {int((nwb.primer_iterado < 0).sum())})")
cociente = []
for entidad, fila in tablas[producto].iterrows():
    i_real = mensual(fila["tasa_EA_%"]/100)
    for n in plazos:
        cociente.append((cuota(P, n, i_real)/P - i_real)/i_real)
print(f"(A/P - i*)/i*: entre {min(cociente):.4f} y {max(cociente):.4f}")
rfa = reg[reg.accion == "regla falsa [0, A/P]"].evaluaciones.mean(); sca = reg[reg.accion == "secante (0, A/P)"].evaluaciones.mean()
print(f"Regla falsa / secante (evaluaciones promedio): {rfa/sca:.2f}")

### Celda 7b · Parte III.3 — Dos "atajos" del agente

(a) Un agente que, para ahorrarse la división, usa f(i) = P·i − A·(1 − (1 + i)⁻ⁿ) en vez de g.
(b) Un agente que para cuando |g(i)| < 0,1 millones de pesos.

**Predice en tu guía antes de correr:** ¿cuánto vale f(0)? ¿Qué tasa reporta el agente (a) si arranca en i₀ = 0? En (b), ¿la oferta de la Celda 6 sigue siendo ilegal?

In [ ]:
P, n, A = 10_000_000, 36, 400_000   # la oferta de la Celda 6
f  = lambda i: P*i - A*(1 - (1 + i)**(-n))
fp = lambda i: P - n*A*(1 + i)**(-n - 1)
print(f"(a) f(0) = {f(0.0)}")
i_m = (n*A/P)**(1/(n + 1)) - 1
print(f"    f'(i) = 0 en i_m = {i_m:.6f}")
for i0 in [0.0, 0.001, 0.005, i_m - 1e-4, i_m + 1e-4, 0.02, A/P]:   # <-- CAMBIA ESTO
    x, it, ev, _ = newton(f, fp, i0)
    print(f"    Newton sobre f desde i0 = {i0:.6f}: converge a i = {x:.6g} en {it} iteraciones  ->  {100*anual(x):.4f}% E.A.")

print("\n(b) Parar cuando |g(i)| < tol_g (g en pesos):")
g, gp = hacer_g(P, n, A)
i_ver = newton(g, gp, 0.0)[0]
for tol_g in [1e5, 1e4, 1e3, 1.0]:    # <-- CAMBIA ESTO
    x = 0.0; k = 0
    while abs(g(x)) >= tol_g:
        x = x - g(x)/gp(x); k += 1
    print(f"    tol_g = {tol_g:>9,.0f} pesos: Newton(i0=0) para en la iteración {k} con {100*anual(x):.4f}% E.A. "
          f"(error en i = {abs(x - i_ver):.2e}) -> {'ILEGAL' if anual(x) > usura['2026-10']/100 else 'legal'}")

print("\n(c) El auditor de UNA evaluación: g es creciente, así que basta el signo de g en la tasa de usura.")
i_us = mensual(usura["2026-10"]/100)
print(f"    g(i_usura) = {g(i_us):,.2f} pesos  ->  la raíz está a la {'DERECHA' if g(i_us) < 0 else 'izquierda'} de la usura")